# RankMixer v2.1 score-oriented experiment

- This notebook is copied from `rankmixer_fair_ablation_v2.ipynb`.
- This version is for metric-oriented experiments, not fair ablation.
- The original v2 notebook must remain untouched.
- All major changes should be controlled by config flags.


# RankMixer 公平性消融（v2）

这个版本把代码结构重排成更接近 `6.ranking.ipynb` 的教学式 notebook，重点保留三条主线：

1. 显式区分 `MODEL_SPARSE_FEATURES`、`MODEL_DENSE_FEATURES`、`META_FEATURES`
2. 把历史序列截断修成 recent-right 版本：`seq[-maxlen:]`、右对齐、`padding=0`
3. 把 RankMixer 升级为 fieldwise token schema + 可学习 `self_attn` token mixing

当前 notebook 只保留 RankMixer 各分支对比：

- `two_logit_jrc` 作为主线
- 2D 主线只保留 `balanced_ge + click` 与 `old_ge + diff`
- `single_logit` 作为 listwise softmax 消融，而不是替代主线
- 不再训练 `MLP baseline` 分支


In [37]:
from __future__ import annotations

import copy
import random
import time
from collections import OrderedDict, defaultdict
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Sequence

import numpy as np
import pandas as pd
from sklearn.metrics import ndcg_score, roc_auc_score
from sklearn.preprocessing import MinMaxScaler
from IPython.display import display

try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    from torch.utils.data import DataLoader, Dataset

    TORCH_AVAILABLE = True
    TORCH_IMPORT_ERROR = None
except ImportError as exc:  # pragma: no cover - environment dependent
    torch = nn = F = DataLoader = Dataset = None
    TORCH_AVAILABLE = False
    TORCH_IMPORT_ERROR = exc

TorchModuleBase = nn.Module if TORCH_AVAILABLE else object
TorchDatasetBase = Dataset if TORCH_AVAILABLE else object

# ============================================================================
# 核心运行配置
# ============================================================================
MODE = "offline"
SEED = 2020
EPOCHS = 5
BATCH_SIZE = 256
LR = 1e-3
TOPK = 5
ALPHA = 0.5

EXPERIMENT_NAME = "rankmixer_v2_score"

USE_HARD_NEGATIVE_TOP64 = False
USE_NN_RANK_SCORE_FEATURES = False
USE_TRAIN_ONLY_VOCAB = False
USE_ARTICLE_SVD = False
USE_CLS_POOLING = False
USE_DENSE_SHORTCUT = False
USE_LISTWISE_BPR_LOSS = False
USE_FINAL_RECALL_SCORE_GRID = False

ARTICLE_SVD_DIM = 16
TRAIN_DATA_VARIANT = "baseline"  # options: "baseline", "top64", "top100"

def log_experiment_config():
    print("Experiment config:")
    print(f"  EXPERIMENT_NAME={EXPERIMENT_NAME}")
    for flag_name in sorted(name for name in globals() if name.startswith("USE_")):
        print(f"  {flag_name}={globals()[flag_name]}")
    print(f"  ARTICLE_SVD_DIM={ARTICLE_SVD_DIM}")
    print(f"  TRAIN_DATA_VARIANT={TRAIN_DATA_VARIANT}")

PROJECT_ROOT = Path.cwd().resolve()
SAVE_PATH = PROJECT_ROOT / "data" / "processed" / "temp_results"
DATA_PATH = PROJECT_ROOT / "data" / "raw" / "news_recommendation"
offline = MODE == "offline"

print("=" * 70)
print("RankMixer fair ablation notebook v2")
print(f"MODE: {MODE}")
print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"SAVE_PATH: {SAVE_PATH}")
print(f"DATA_PATH: {DATA_PATH}")
print("=" * 70)


RankMixer fair ablation notebook v2
MODE: offline
PROJECT_ROOT: /Users/lixiang/Desktop/funrec-new-rec
SAVE_PATH: /Users/lixiang/Desktop/funrec-new-rec/data/processed/temp_results
DATA_PATH: /Users/lixiang/Desktop/funrec-new-rec/data/raw/news_recommendation


## 实验设置与特征边界


In [38]:
from __future__ import annotations

# ============================================================================
# 特征边界、实验场景与数据结构
# ============================================================================
MODEL_SPARSE_FEATURES = [
    "click_article_id",
    "category_id",
    "click_environment",
    "click_deviceGroup",
    "click_os",
    "click_country",
    "click_region",
    "click_referrer_type",
]

MODEL_DENSE_FEATURES = [
    "sim0",
    "time_diff0",
    "word_diff0",
    "sim_max",
    "sim_min",
    "sim_sum",
    "sim_mean",
    "score",
    "rank",
    "click_size",
    "time_diff_mean",
    "active_level",
    "created_at_ts",
    "user_time_hob1",
    "user_time_hob2",
    "words_hbo",
    "words_count",
    "is_cat_hab",
    "article_hot_level",
    "article_user_num",
    "article_time_diff_mean",
]

META_INT_FEATURES = ["query_id"]
META_FLOAT_FEATURES = ["sample_prob"]
META_FEATURES = list(META_INT_FEATURES)
assert "sample_prob" not in MODEL_SPARSE_FEATURES, "sample_prob must stay out of MODEL_SPARSE_FEATURES."
assert "sample_prob" not in MODEL_DENSE_FEATURES, "sample_prob must stay out of MODEL_DENSE_FEATURES."
ALL_SPARSE_DIAGNOSTIC_FEATURES = ["user_id"] + MODEL_SPARSE_FEATURES
HISTORY_FEATURE = "hist_click_article_id"

DEFAULT_EMB_DIM = 8
DEFAULT_MAX_HIST_LEN = 50
DEFAULT_MODEL_CONFIG = {
    "emb_dim": DEFAULT_EMB_DIM,
    "max_len": DEFAULT_MAX_HIST_LEN,
    "shared_dnn_units": [256, 128],
    "logit_head_units": [64],
    "attention_hidden_units": [64, 32],
    "dropout_rate": 0.1,
}
DEFAULT_RANKMIXER_SEARCH_SPACE = {
    "token_dim": [24, 32, 40, 48, 56, 64, 72, 80, 96, 112, 128],
    "num_layers": [1, 2, 3],
    "expansion_ratio": [2, 3, 4],
    "num_heads": [2, 4, 6, 8],
}
PARAM_RATIO_TOLERANCE = 0.10

TWO_LOGIT_PROTOCOLS = [
    {"loss_type": "balanced_ge", "score_mode": "click"},
    {"loss_type": "old_ge", "score_mode": "diff"},
]
SINGLE_LOGIT_PROTOCOLS = [
    {"loss_type": "query_softmax_ce", "score_mode": "scalar"},
]


def ensure_torch_available():
    if not TORCH_AVAILABLE:  # pragma: no cover - environment dependent
        raise ImportError(
            "rankmixer_fair_ablation_v2 requires PyTorch. Install torch and rerun."
        ) from TORCH_IMPORT_ERROR


def get_default_device():
    ensure_torch_available()
    if torch.cuda.is_available():
        return torch.device("cuda")
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


def clear_torch_cache():
    if not TORCH_AVAILABLE:
        return
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        return
    if hasattr(torch, "mps") and hasattr(torch.mps, "empty_cache"):
        torch.mps.empty_cache()


def set_global_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    if TORCH_AVAILABLE:
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)


@dataclass(frozen=True)
class FeaturePreset:
    name: str
    model_sparse_features: List[str]
    model_dense_features: List[str]
    meta_features: List[str] = field(default_factory=lambda: list(META_FEATURES))
    description: str = ""


@dataclass(frozen=True)
class ScenarioSpec:
    name: str
    feature_preset: FeaturePreset
    head_type: str
    protocols: List[Dict[str, str]]
    notes: str
    is_mainline: bool


@dataclass
class LoadedFeatureFrames:
    trn_user_item_feats_df: pd.DataFrame
    val_user_item_feats_df: Optional[pd.DataFrame]
    tst_user_item_feats_df: pd.DataFrame
    click_hist_all: pd.DataFrame
    offline: bool


@dataclass
class MergedFeatureFrames:
    trn_df: pd.DataFrame
    val_df: Optional[pd.DataFrame]
    tst_df: pd.DataFrame
    his_behavior_df: pd.DataFrame
    val_hit_mask: Optional[pd.Series]
    val_df_hit: Optional[pd.DataFrame]
    offline: bool


@dataclass
class PreparedRankingData:
    trn_df: pd.DataFrame
    val_df: Optional[pd.DataFrame]
    tst_df: pd.DataFrame
    val_hit_mask: Optional[pd.Series]
    val_df_hit: Optional[pd.DataFrame]
    sparse_maps: Dict[str, Dict[int, int]]
    sparse_vocab_sizes: Dict[str, int]
    mm_scalers: Dict[str, MinMaxScaler]
    x_trn: Dict[str, np.ndarray]
    y_trn: np.ndarray
    x_val: Optional[Dict[str, np.ndarray]]
    y_val: Optional[np.ndarray]
    x_tst: Optional[Dict[str, np.ndarray]]
    dense_input_dim: int
    emb_dim: int
    max_len: int
    feature_preset: FeaturePreset


@dataclass
class PreparedScenario:
    spec: ScenarioSpec
    prepared_data: PreparedRankingData
    model_config: Dict[str, Any]
    reference_params: int
    selected_rankmixer_config: Dict[str, Any]
    selected_rankmixer_params: int
    rankmixer_search_result: Dict[str, Any]
    token_schema_df: pd.DataFrame


@dataclass
class PreparedAblationSuite:
    merged_frames: MergedFeatureFrames
    sparse_feature_diagnostics_df: pd.DataFrame
    scenarios: List[PreparedScenario]


def build_default_feature_presets() -> Dict[str, FeaturePreset]:
    fair_main = FeaturePreset(
        name="fair_main",
        model_sparse_features=list(MODEL_SPARSE_FEATURES),
        model_dense_features=list(MODEL_DENSE_FEATURES),
        meta_features=list(META_FEATURES),
        description=(
            "Main experiment: remove user_id from learned sparse features, keep query_id only as metadata, "
            "include created_at_ts in dense features, and use recent-right history truncation."
        ),
    )
    user_id_control = FeaturePreset(
        name="user_id_control",
        model_sparse_features=["user_id"] + list(MODEL_SPARSE_FEATURES),
        model_dense_features=list(MODEL_DENSE_FEATURES),
        meta_features=list(META_FEATURES),
        description=(
            "Control ablation: keep user_id as a learned sparse feature while query_id still drives grouping, "
            "to make the user-disjoint leakage risk explicit."
        ),
    )
    return {
        fair_main.name: fair_main,
        user_id_control.name: user_id_control,
    }


def build_default_scenarios() -> List[ScenarioSpec]:
    presets = build_default_feature_presets()
    return [
        ScenarioSpec(
            name="fair_main_two_logit_jrc",
            feature_preset=presets["fair_main"],
            head_type="two_logit_jrc",
            protocols=copy.deepcopy(TWO_LOGIT_PROTOCOLS),
            notes="Primary 2D-logit JRC line with balanced_ge + click and old_ge + diff.",
            is_mainline=True,
        ),
        ScenarioSpec(
            name="user_id_control_two_logit_jrc",
            feature_preset=presets["user_id_control"],
            head_type="two_logit_jrc",
            protocols=copy.deepcopy(TWO_LOGIT_PROTOCOLS),
            notes="Control branch that restores user_id as a learned sparse feature under the same RankMixer-only protocols.",
            is_mainline=False,
        ),
        ScenarioSpec(
            name="fair_main_single_logit",
            feature_preset=presets["fair_main"],
            head_type="single_logit",
            protocols=copy.deepcopy(SINGLE_LOGIT_PROTOCOLS),
            notes="Single-logit listwise softmax ablation. This is an ablation on top of the 2D mainline, not a replacement.",
            is_mainline=False,
        ),
    ]


In [39]:
from __future__ import annotations

# query_id 是 raw user_id 的只读镜像，只用于 query 分组。
set_global_seed(SEED)
print(f"PyTorch available: {TORCH_AVAILABLE}")
if TORCH_AVAILABLE:
    DEVICE = get_default_device()
    print(f"Device: {DEVICE}")
else:
    DEVICE = None
    print("PyTorch is required for the parameter-search and training cells below.")

log_experiment_config()

feature_boundary_df = pd.DataFrame(
    [
        {"feature_group": "MODEL_SPARSE_FEATURES", "features": ", ".join(MODEL_SPARSE_FEATURES)},
        {"feature_group": "MODEL_DENSE_FEATURES", "features": ", ".join(MODEL_DENSE_FEATURES)},
        {"feature_group": "META_INT_FEATURES", "features": ", ".join(META_INT_FEATURES)},
        {"feature_group": "META_FLOAT_FEATURES", "features": ", ".join(META_FLOAT_FEATURES)},
        {"feature_group": "META_FEATURES", "features": ", ".join(META_FEATURES)},
    ]
)
display(feature_boundary_df)
print("query_id = raw user_id, and query_id is only used for grouped batching / grouped loss / grouped metrics.")
print("query_id never enters learned sparse embeddings or model forward features.")
print("sample_prob is metadata-only and is not part of sparse or dense model feature groups.")


PyTorch available: True
Device: mps


,feature_group,features
0,MODEL_SPARSE_FEATURES,"click_article_id, category_id, click_environme..."
1,MODEL_DENSE_FEATURES,"sim0, time_diff0, word_diff0, sim_max, sim_min..."
2,META_FEATURES,query_id


query_id = raw user_id, and query_id is only used for grouped batching / grouped loss / grouped metrics.
query_id never enters learned sparse embeddings or model forward features.


## 读取特征与构造 Query 级样本


In [40]:
from __future__ import annotations

# ============================================================================
# 读取原始特征，并把 raw user_id 显式保存为 query_id
# ============================================================================
def resolve_train_feature_path(save_path: Path, train_data_variant: str) -> Path:
    variant_to_filename = {
        "baseline": "trn_user_item_feats_df_all.csv",
        "top64": "trn_user_item_feats_df_all_rankmixer_v2_top64.csv",
        "top100": "trn_user_item_feats_df_all_rankmixer_v2_top100.csv",
    }
    if train_data_variant not in variant_to_filename:
        raise ValueError(
            f"Unsupported TRAIN_DATA_VARIANT: {train_data_variant}. "
            "Expected one of ['baseline', 'top64', 'top100']."
        )

    train_path = save_path / variant_to_filename[train_data_variant]
    if train_path.exists():
        return train_path

    if train_data_variant == "top64":
        raise FileNotFoundError(
            f"Training feature file not found: {train_path}. "
            "TRAIN_DATA_VARIANT='top64' requires running 5.feature_engineering_all.ipynb "
            "with NEG_SAMPLE_MODE='rankmixer_v2_top64' first."
        )
    if train_data_variant == "top100":
        raise FileNotFoundError(
            f"Training feature file not found: {train_path}. "
            "Please generate the top100 training feature file first."
        )
    raise FileNotFoundError(f"Training feature file not found: {train_path}")


def _normalize_sample_prob_column(df: Optional[pd.DataFrame]) -> Optional[pd.DataFrame]:
    if df is None:
        return None
    df = df.copy()
    if "sample_prob" not in df.columns:
        df["sample_prob"] = 1.0
    sample_prob = pd.to_numeric(df["sample_prob"], errors="coerce")
    sample_prob = sample_prob.replace([np.inf, -np.inf], np.nan)
    sample_prob = sample_prob.fillna(1.0).clip(1e-4, 1.0).astype("float32")
    df["sample_prob"] = sample_prob
    return df


def ensure_sampling_metadata_for_loaded_frames(
    trn_df: pd.DataFrame,
    val_df: Optional[pd.DataFrame],
    tst_df: pd.DataFrame,
    train_data_variant: str,
):
    trn_df = trn_df.copy()
    val_df = val_df.copy() if val_df is not None else None
    tst_df = tst_df.copy()

    if train_data_variant != "baseline":
        required_cols = ["sample_prob", "neg_bucket"]
        missing_cols = [col for col in required_cols if col not in trn_df.columns]
        if missing_cols:
            raise KeyError(
                f"Training variant '{train_data_variant}' requires train columns {required_cols}. "
                f"Missing columns: {missing_cols}"
            )
    elif "sample_prob" not in trn_df.columns:
        trn_df["sample_prob"] = 1.0

    trn_df = _normalize_sample_prob_column(trn_df)
    val_df = _normalize_sample_prob_column(val_df)
    tst_df = _normalize_sample_prob_column(tst_df)

    print("Train sample_prob.describe():")
    print(trn_df["sample_prob"].describe())
    if "neg_bucket" in trn_df.columns:
        print("Train neg_bucket.value_counts(dropna=False):")
        print(trn_df["neg_bucket"].value_counts(dropna=False))

    return trn_df, val_df, tst_df


def load_feature_frames(
    save_path: Path,
    offline: bool = True,
    train_data_variant: str = TRAIN_DATA_VARIANT,
) -> LoadedFeatureFrames:
    trn_path = resolve_train_feature_path(save_path, train_data_variant)
    val_path = save_path / "val_user_item_feats_df_all.csv"
    tst_path = save_path / "tst_user_item_feats_df_all.csv"

    print(f"Selected train data variant: {train_data_variant}")
    print(f"Train feature path: {trn_path}")
    print(f"Val feature path: {val_path}")
    print(f"Test feature path: {tst_path}")

    trn_df = pd.read_csv(trn_path)
    trn_df["click_article_id"] = trn_df["click_article_id"].astype(int)

    if offline:
        val_df = pd.read_csv(val_path)
        val_df["click_article_id"] = val_df["click_article_id"].astype(int)
    else:
        val_df = None

    tst_df = pd.read_csv(tst_path)
    if len(tst_df) > 0:
        tst_df["click_article_id"] = tst_df["click_article_id"].astype(int)
    if "label" in tst_df.columns:
        del tst_df["label"]

    trn_df, val_df, tst_df = ensure_sampling_metadata_for_loaded_frames(
        trn_df,
        val_df,
        tst_df,
        train_data_variant=train_data_variant,
    )

    click_hist_all = pd.read_csv(save_path / "click_hist_all.csv")
    return LoadedFeatureFrames(
        trn_user_item_feats_df=trn_df,
        val_user_item_feats_df=val_df,
        tst_user_item_feats_df=tst_df,
        click_hist_all=click_hist_all,
        offline=offline,
    )


def add_query_id_column(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["query_id"] = df["user_id"].astype("int64")
    return df


def build_history_behavior(click_hist_all: pd.DataFrame) -> pd.DataFrame:
    sort_cols = ["user_id"]
    if "click_timestamp" in click_hist_all.columns:
        sort_cols.append("click_timestamp")
    # 先按点击时间排序，再按用户聚合，保证历史序列的时间顺序稳定。
    hist_click = (
        click_hist_all.sort_values(sort_cols, kind="mergesort")[["user_id", "click_article_id"]]
        .groupby("user_id", sort=False)
        .agg(list)
        .reset_index()
    )
    his_behavior_df = pd.DataFrame()
    his_behavior_df["user_id"] = hist_click["user_id"].astype("int64")
    his_behavior_df["query_id"] = his_behavior_df["user_id"]
    his_behavior_df[HISTORY_FEATURE] = hist_click["click_article_id"]
    return his_behavior_df


def merge_history_frames(loaded_frames: LoadedFeatureFrames) -> MergedFeatureFrames:
    his_behavior_df = build_history_behavior(loaded_frames.click_hist_all)

    trn_base = add_query_id_column(loaded_frames.trn_user_item_feats_df)
    trn_df = trn_base.merge(his_behavior_df[["user_id", HISTORY_FEATURE]], on="user_id")

    if loaded_frames.offline and loaded_frames.val_user_item_feats_df is not None:
        val_base = add_query_id_column(loaded_frames.val_user_item_feats_df)
        val_df = val_base.merge(his_behavior_df[["user_id", HISTORY_FEATURE]], on="user_id")
    else:
        val_df = None

    if len(loaded_frames.tst_user_item_feats_df) > 0:
        tst_base = add_query_id_column(loaded_frames.tst_user_item_feats_df)
        tst_df = tst_base.merge(his_behavior_df[["user_id", HISTORY_FEATURE]], on="user_id")
    else:
        tst_df = add_query_id_column(loaded_frames.tst_user_item_feats_df.copy())

    if loaded_frames.offline and val_df is not None:
        val_hit_mask = val_df.groupby("query_id")["label"].transform("max") == 1
        val_df_hit = val_df[val_hit_mask].copy()
    else:
        val_hit_mask = None
        val_df_hit = None

    return MergedFeatureFrames(
        trn_df=trn_df,
        val_df=val_df,
        tst_df=tst_df,
        his_behavior_df=his_behavior_df,
        val_hit_mask=val_hit_mask,
        val_df_hit=val_df_hit,
        offline=loaded_frames.offline,
    )


In [41]:
from __future__ import annotations

# ============================================================================
# 诊断 sparse overlap，并把表格转成模型输入
# ============================================================================
def compute_sparse_feature_overlap_diagnostics(
    trn_df: pd.DataFrame,
    val_df: Optional[pd.DataFrame],
    sparse_features: Sequence[str] = ALL_SPARSE_DIAGNOSTIC_FEATURES,
) -> pd.DataFrame:
    rows: List[Dict[str, Any]] = []
    for feat in sparse_features:
        train_values = set(trn_df[feat].dropna().astype("int64").tolist())
        val_values = set(val_df[feat].dropna().astype("int64").tolist()) if val_df is not None else set()
        overlap_count = len(train_values & val_values)
        val_only_count = len(val_values - train_values)
        val_unique = len(val_values)
        rows.append(
            {
                "sparse_feature": feat,
                "train_unique": len(train_values),
                "val_unique": val_unique,
                "overlap_count": overlap_count,
                "val_only_count": val_only_count,
                "val_only_rate": float(val_only_count / val_unique) if val_unique > 0 else float("nan"),
            }
        )
    return pd.DataFrame(rows).sort_values(
        ["val_only_rate", "val_only_count", "sparse_feature"],
        ascending=[False, False, True],
    ).reset_index(drop=True)


def normalize_dense_features(
    trn_df: pd.DataFrame,
    val_df: Optional[pd.DataFrame],
    tst_df: pd.DataFrame,
    dense_features: Sequence[str],
) -> Dict[str, MinMaxScaler]:
    mm_scalers: Dict[str, MinMaxScaler] = {}
    for feat in dense_features:
        scaler = MinMaxScaler()
        trn_df[feat] = scaler.fit_transform(trn_df[[feat]])
        if val_df is not None:
            val_df[feat] = scaler.transform(val_df[[feat]])
        if len(tst_df) > 0:
            tst_df[feat] = scaler.transform(tst_df[[feat]])
        mm_scalers[feat] = scaler
    return mm_scalers


def build_sparse_maps(
    frames: Sequence[pd.DataFrame],
    sparse_features: Sequence[str],
    history_feature: str = HISTORY_FEATURE,
) -> Dict[str, Dict[int, int]]:
    sparse_maps: Dict[str, Dict[int, int]] = {}
    for feat in sparse_features:
        vals = pd.concat([frame[feat].astype("int64") for frame in frames], ignore_index=True)
        if feat == "click_article_id":
            hist_vals: List[int] = []
            for frame in frames:
                for seq in frame[history_feature]:
                    if isinstance(seq, (list, np.ndarray)):
                        hist_vals.extend(int(x) for x in seq)
            if hist_vals:
                vals = pd.concat([vals, pd.Series(hist_vals, dtype="int64")], ignore_index=True)
        _, uniques = pd.factorize(vals, sort=False)
        mapping = {int(val): int(idx) for idx, val in enumerate(uniques)}
        if feat == "click_article_id":
            mapping = {key: value + 1 for key, value in mapping.items()}
        sparse_maps[feat] = mapping
    return sparse_maps


def _vocab_size(mapping: Dict[int, int]) -> int:
    return max(mapping.values()) + 1 if mapping else 1


def pad_sequences_np(
    sequences: Sequence[Sequence[int]],
    maxlen: int,
    value: int = 0,
    keep: str = "recent",
    align: str = "right",
) -> np.ndarray:
    padded = np.full((len(sequences), maxlen), value, dtype=np.int64)
    # recent 模式取 seq[-maxlen:]，并在张量中右对齐，padding 固定为 0。
    for idx, seq in enumerate(sequences):
        if len(seq) == 0:
            continue
        if keep == "recent":
            trunc = list(seq)[-maxlen:]
        elif keep == "earliest":
            trunc = list(seq)[:maxlen]
        else:  # pragma: no cover - defensive
            raise ValueError(f"Unknown keep mode: {keep}")
        trunc_arr = np.asarray(trunc, dtype=np.int64)
        if align == "right":
            padded[idx, -len(trunc_arr) :] = trunc_arr
        elif align == "left":
            padded[idx, : len(trunc_arr)] = trunc_arr
        else:  # pragma: no cover - defensive
            raise ValueError(f"Unknown align mode: {align}")
    return padded


def build_model_input(
    df: pd.DataFrame,
    sparse_maps: Dict[str, Dict[int, int]],
    feature_preset: FeaturePreset,
    max_len: int = DEFAULT_MAX_HIST_LEN,
) -> Dict[str, np.ndarray]:
    model_input: Dict[str, np.ndarray] = {}
    for feat in feature_preset.model_sparse_features:
        model_input[feat] = (
            df[feat].astype("int64").map(sparse_maps[feat]).fillna(0).astype("int64").values
        )
    for feat in feature_preset.model_dense_features:
        model_input[feat] = df[feat].astype("float32").values
    for feat in feature_preset.meta_features:
        model_input[feat] = df[feat].astype("int64").values
    for feat in META_FLOAT_FEATURES:
        if feat in df.columns:
            model_input[feat] = df[feat].astype("float32").values

    # history 序列共用 click_article_id 的映射，保证目标 item 与历史 item 使用同一套 embedding。
    click_mapping = sparse_maps["click_article_id"]
    raw_seq_list = df[HISTORY_FEATURE].tolist()
    mapped_seq: List[List[int]] = []
    for seq in raw_seq_list:
        if isinstance(seq, (list, np.ndarray)):
            mapped_seq.append([int(click_mapping.get(int(x), 0)) if int(x) != 0 else 0 for x in seq])
        else:
            mapped_seq.append([])
    model_input[HISTORY_FEATURE] = pad_sequences_np(
        mapped_seq,
        maxlen=max_len,
        value=0,
        keep="recent",
        align="right",
    )
    return model_input


def prepare_model_inputs(
    merged_frames: MergedFeatureFrames,
    feature_preset: FeaturePreset,
    emb_dim: int = DEFAULT_EMB_DIM,
    max_len: int = DEFAULT_MAX_HIST_LEN,
) -> PreparedRankingData:
    trn_df = merged_frames.trn_df.copy()
    val_df = merged_frames.val_df.copy() if merged_frames.val_df is not None else None
    tst_df = merged_frames.tst_df.copy()

    mm_scalers = normalize_dense_features(
        trn_df,
        val_df,
        tst_df,
        dense_features=feature_preset.model_dense_features,
    )
    all_frames = [trn_df] + ([val_df] if val_df is not None else []) + ([tst_df] if len(tst_df) > 0 else [])
    sparse_maps = build_sparse_maps(all_frames, sparse_features=feature_preset.model_sparse_features)
    sparse_vocab_sizes = {
        feat: _vocab_size(sparse_maps[feat]) for feat in feature_preset.model_sparse_features
    }

    x_trn = build_model_input(trn_df, sparse_maps, feature_preset, max_len=max_len)
    y_trn = trn_df["label"].values.astype("float32")

    if val_df is not None:
        x_val = build_model_input(val_df, sparse_maps, feature_preset, max_len=max_len)
        y_val = val_df["label"].values.astype("float32")
    else:
        x_val, y_val = None, None

    if len(tst_df) > 0:
        x_tst = build_model_input(tst_df, sparse_maps, feature_preset, max_len=max_len)
    else:
        x_tst = None

    return PreparedRankingData(
        trn_df=trn_df,
        val_df=val_df,
        tst_df=tst_df,
        val_hit_mask=merged_frames.val_hit_mask,
        val_df_hit=merged_frames.val_df_hit,
        sparse_maps=sparse_maps,
        sparse_vocab_sizes=sparse_vocab_sizes,
        mm_scalers=mm_scalers,
        x_trn=x_trn,
        y_trn=y_trn,
        x_val=x_val,
        y_val=y_val,
        x_tst=x_tst,
        dense_input_dim=len(feature_preset.model_dense_features),
        emb_dim=emb_dim,
        max_len=max_len,
        feature_preset=feature_preset,
    )


In [42]:
from __future__ import annotations

loaded_frames = load_feature_frames(SAVE_PATH, offline=offline, train_data_variant=TRAIN_DATA_VARIANT)

print(f"Train feature frame: {loaded_frames.trn_user_item_feats_df.shape}")
if loaded_frames.val_user_item_feats_df is not None:
    print(f"Val feature frame: {loaded_frames.val_user_item_feats_df.shape}")
else:
    print("Val feature frame: None")
print(f"Test feature frame: {loaded_frames.tst_user_item_feats_df.shape}")
print(f"Click history frame: {loaded_frames.click_hist_all.shape}")


Train feature frame: (1426880, 31)
Val feature frame: (8000000, 31)
Test feature frame: (0, 31)
Click history frame: (912623, 9)


In [ ]:
from __future__ import annotations

print(f"train contains sample_prob: {'sample_prob' in loaded_frames.trn_user_item_feats_df.columns}")
print(
    f"val contains sample_prob: {loaded_frames.val_user_item_feats_df is not None and 'sample_prob' in loaded_frames.val_user_item_feats_df.columns}"
)
print(f"test contains sample_prob: {'sample_prob' in loaded_frames.tst_user_item_feats_df.columns}")
print(f"sample_prob absent from MODEL_SPARSE_FEATURES: {'sample_prob' not in MODEL_SPARSE_FEATURES}")
print(f"sample_prob absent from MODEL_DENSE_FEATURES: {'sample_prob' not in MODEL_DENSE_FEATURES}")


In [43]:
from __future__ import annotations

merged_frames = merge_history_frames(loaded_frames)
sparse_feature_diagnostics_df = compute_sparse_feature_overlap_diagnostics(
    merged_frames.trn_df,
    merged_frames.val_df,
)

print(f"Merged train frame: {merged_frames.trn_df.shape}")
if merged_frames.val_df is not None:
    print(f"Merged val frame: {merged_frames.val_df.shape}")
else:
    print("Merged val frame: None")
print(f"Merged test frame: {merged_frames.tst_df.shape}")
print(f"History users: {merged_frames.his_behavior_df.shape}")
if merged_frames.val_df_hit is not None:
    print(f"Val hit queries: {merged_frames.val_df_hit['query_id'].nunique()}")
    print(f"Val hit samples: {len(merged_frames.val_df_hit)}")
print(f"query_id matches raw user_id in train: {(merged_frames.trn_df['query_id'] == merged_frames.trn_df['user_id']).all()}")
if merged_frames.val_df is not None:
    print(f"query_id matches raw user_id in val: {(merged_frames.val_df['query_id'] == merged_frames.val_df['user_id']).all()}")

display(sparse_feature_diagnostics_df)


Merged train frame: (1426880, 33)
Merged val frame: (8000000, 33)
Merged test frame: (0, 32)
History users: (200000, 3)
Val hit queries: 27382
Val hit samples: 5476400
query_id matches raw user_id in train: True
query_id matches raw user_id in val: True


,sparse_feature,train_unique,val_unique,overlap_count,val_only_count,val_only_rate
0,user_id,109760,40000,0,40000,1.000000
1,click_article_id,18626,20283,18427,1856,0.091505
2,category_id,244,247,243,4,0.016194
3,click_country,11,11,11,0,0.000000
4,click_deviceGroup,4,4,4,0,0.000000
5,click_environment,3,3,3,0,0.000000
6,click_os,8,8,8,0,0.000000
7,click_referrer_type,7,7,7,0,0.000000
8,click_region,28,27,27,0,0.000000


## 模型结构：DIN 编码器与 RankMixer


In [44]:
from __future__ import annotations

# ============================================================================
# 基础模型组件：MLP、DIN pooling、统一输入编码器
# ============================================================================
def make_mlp(
    input_dim: int,
    hidden_units: Sequence[int],
    dropout_rate: float = 0.0,
    output_dim: Optional[int] = None,
    output_activation: Optional[str] = None,
):
    ensure_torch_available()
    layers: List[nn.Module] = []
    prev_dim = input_dim
    for units in hidden_units:
        layers.append(nn.Linear(prev_dim, units))
        layers.append(nn.ReLU())
        if dropout_rate > 0:
            layers.append(nn.Dropout(dropout_rate))
        prev_dim = units
    if output_dim is not None:
        layers.append(nn.Linear(prev_dim, output_dim))
        prev_dim = output_dim
        if output_activation == "sigmoid":
            layers.append(nn.Sigmoid())
    return nn.Sequential(*layers), prev_dim


class DINAttentionPooling(TorchModuleBase):
    def __init__(self, emb_dim: int, hidden_units: Sequence[int]):
        super().__init__()
        self.att_mlp, _ = make_mlp(
            emb_dim * 4,
            hidden_units,
            dropout_rate=0.0,
            output_dim=1,
            output_activation=None,
        )

    def forward(self, query_emb: torch.Tensor, hist_emb: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        query = query_emb.unsqueeze(1).expand_as(hist_emb)
        att_input = torch.cat([query, hist_emb, query - hist_emb, query * hist_emb], dim=-1)
        scores = self.att_mlp(att_input).squeeze(-1)

        all_pad = ~mask.any(dim=1)
        scores = scores.masked_fill(~mask, -1e9)
        scores = scores.masked_fill(all_pad.unsqueeze(1), 0.0)

        weights = torch.softmax(scores, dim=1)
        weights = weights * mask.float()
        denom = weights.sum(dim=1, keepdim=True).clamp_min(1e-8)
        weights = weights / denom

        pooled = torch.bmm(weights.unsqueeze(1), hist_emb).squeeze(1)
        pooled = pooled.masked_fill(all_pad.unsqueeze(1), 0.0)
        return pooled


class JRCBackboneInputEncoder(TorchModuleBase):
    def __init__(self, sparse_vocab_sizes: Dict[str, int], dense_features: Sequence[str], model_config: Dict[str, Any]):
        super().__init__()
        self.sparse_features = list(sparse_vocab_sizes.keys())
        self.dense_features = list(dense_features)
        self.emb_dim = model_config["emb_dim"]
        self.sparse_embeddings = nn.ModuleDict()
        for feat, vocab_size in sparse_vocab_sizes.items():
            if feat == "click_article_id":
                self.sparse_embeddings[feat] = nn.Embedding(vocab_size, self.emb_dim, padding_idx=0)
            else:
                self.sparse_embeddings[feat] = nn.Embedding(vocab_size, self.emb_dim)
        self.din_pooling = DINAttentionPooling(
            self.emb_dim,
            model_config.get("attention_hidden_units", [64, 32]),
        )

    def forward(self, batch: Dict[str, torch.Tensor]) -> Dict[str, Any]:
        sparse_emb_dict = {
            feat: self.sparse_embeddings[feat](batch[feat]) for feat in self.sparse_features
        }
        dense_tensor = torch.stack([batch[feat] for feat in self.dense_features], dim=1)

        target_emb = sparse_emb_dict["click_article_id"]
        hist_ids = batch[HISTORY_FEATURE]
        hist_emb = self.sparse_embeddings["click_article_id"](hist_ids)
        hist_mask = hist_ids.ne(0)
        hist_repr = self.din_pooling(target_emb, hist_emb, hist_mask)

        flat_sparse = [sparse_emb_dict[feat] for feat in self.sparse_features]
        flat_input = torch.cat(flat_sparse + [dense_tensor, hist_repr], dim=1)
        return {
            "sparse_embeddings": sparse_emb_dict,
            "dense_tensor": dense_tensor,
            "hist_repr": hist_repr,
            "flat_input": flat_input,
        }


In [45]:
from __future__ import annotations

# ============================================================================
# RankMixer：fieldwise token schema + 可学习 self-attention token mixing
# ============================================================================
SPARSE_TOKEN_ORDER = [
    "user_id",
    "click_article_id",
    "category_id",
    "click_environment",
    "click_deviceGroup",
    "click_os",
    "click_country",
    "click_region",
    "click_referrer_type",
]

DENSE_TOKEN_GROUPS = OrderedDict(
    [
        (
            "recall_similarity_dense",
            ["sim0", "time_diff0", "word_diff0", "sim_max", "sim_min", "sim_sum", "sim_mean"],
        ),
        (
            "ranking_meta_dense",
            ["score", "rank", "click_size", "time_diff_mean", "active_level", "created_at_ts"],
        ),
        (
            "user_article_profile_dense",
            [
                "user_time_hob1",
                "user_time_hob2",
                "words_hbo",
                "words_count",
                "is_cat_hab",
                "article_hot_level",
                "article_user_num",
                "article_time_diff_mean",
            ],
        ),
    ]
)

assert all(
    "sample_prob" not in features for features in DENSE_TOKEN_GROUPS.values()
), "sample_prob must stay out of DENSE_TOKEN_GROUPS."


def build_token_specs(feature_preset: FeaturePreset) -> List[Dict[str, Any]]:
    specs: List[Dict[str, Any]] = []
    remaining_sparse = set(feature_preset.model_sparse_features)
    for feat in SPARSE_TOKEN_ORDER:
        if feat in remaining_sparse:
            token_name = feat if feat != "user_id" else "user_id_sparse_control"
            specs.append({"name": token_name, "kind": "sparse", "features": [feat]})
            remaining_sparse.remove(feat)
    if remaining_sparse:
        for feat in sorted(remaining_sparse):
            specs.append({"name": feat, "kind": "sparse", "features": [feat]})

    remaining_dense = set(feature_preset.model_dense_features)
    for token_name, features in DENSE_TOKEN_GROUPS.items():
        included = [feat for feat in features if feat in remaining_dense]
        if included:
            specs.append({"name": token_name, "kind": "dense", "features": included})
            remaining_dense -= set(included)
    if remaining_dense:
        specs.append({"name": "dense_residual", "kind": "dense", "features": sorted(remaining_dense)})
        remaining_dense.clear()

    specs.append({"name": "history_token", "kind": "history", "features": [HISTORY_FEATURE]})
    return specs


def make_token_schema_dataframe(feature_preset: FeaturePreset) -> pd.DataFrame:
    rows: List[Dict[str, Any]] = []
    for idx, spec in enumerate(build_token_specs(feature_preset), start=1):
        rows.append(
            {
                "token_index": idx,
                "token_name": spec["name"],
                "kind": spec["kind"],
                "features": ", ".join(spec["features"]),
            }
        )
    return pd.DataFrame(rows)


class FieldwiseTokenization(TorchModuleBase):
    def __init__(self, emb_dim: int, token_dim: int, feature_preset: FeaturePreset, verbose: bool = False):
        super().__init__()
        self.emb_dim = emb_dim
        self.token_dim = token_dim
        self.feature_preset = feature_preset
        self.token_specs = build_token_specs(feature_preset)
        self.projections = nn.ModuleDict()
        self.token_input_dims: Dict[str, int] = {}
        self._validate_coverage()

        for spec in self.token_specs:
            if spec["kind"] == "sparse":
                input_dim = len(spec["features"]) * emb_dim
            elif spec["kind"] == "dense":
                input_dim = len(spec["features"])
            elif spec["kind"] == "history":
                input_dim = emb_dim
            else:  # pragma: no cover - defensive
                raise ValueError(f"Unknown token kind: {spec['kind']}")
            self.token_input_dims[spec["name"]] = input_dim
            self.projections[spec["name"]] = nn.Linear(input_dim, token_dim)

        if verbose:
            self.print_summary()

    def _validate_coverage(self):
        sparse_consumed: List[str] = []
        dense_consumed: List[str] = []
        for spec in self.token_specs:
            if spec["kind"] == "sparse":
                sparse_consumed.extend(spec["features"])
            elif spec["kind"] == "dense":
                dense_consumed.extend(spec["features"])
        assert len(sparse_consumed) == len(set(sparse_consumed)), "Sparse token specs contain duplicates."
        assert len(dense_consumed) == len(set(dense_consumed)), "Dense token specs contain duplicates."
        assert set(sparse_consumed) == set(
            self.feature_preset.model_sparse_features
        ), "Sparse features are not consumed exactly once."
        assert set(dense_consumed) == set(
            self.feature_preset.model_dense_features
        ), "Dense features are not consumed exactly once."

    def print_summary(self):
        print("FieldwiseTokenization summary:")
        for spec in self.token_specs:
            print(
                f"  {spec['name']}: kind={spec['kind']} "
                f"features={spec['features']} input_dim={self.token_input_dims[spec['name']]}"
            )
        print(f"  num_tokens={len(self.token_specs)}")

    def forward(
        self,
        batch: Dict[str, torch.Tensor],
        sparse_embeddings: Dict[str, torch.Tensor],
        hist_repr: torch.Tensor,
    ) -> torch.Tensor:
        # 每个 token 对应一个语义字段组，最后堆成 [B, T, D] 送入 RankMixer。
        tokens: List[torch.Tensor] = []
        for spec in self.token_specs:
            if spec["kind"] == "sparse":
                token_input = torch.cat([sparse_embeddings[feat] for feat in spec["features"]], dim=1)
            elif spec["kind"] == "dense":
                token_input = torch.stack([batch[feat] for feat in spec["features"]], dim=1)
            else:
                token_input = hist_repr
            tokens.append(self.projections[spec["name"]](token_input))
        return torch.stack(tokens, dim=1)


class SelfAttentionMixerLayer(TorchModuleBase):
    def __init__(self, token_dim: int, num_heads: int, expansion_ratio: int = 4, dropout_rate: float = 0.0):
        super().__init__()
        assert token_dim % num_heads == 0, "token_dim % num_heads must equal 0."
        self.self_attn = nn.MultiheadAttention(
            embed_dim=token_dim,
            num_heads=num_heads,
            dropout=dropout_rate,
            batch_first=True,
        )
        hidden_dim = token_dim * expansion_ratio
        self.norm1 = nn.LayerNorm(token_dim)
        self.norm2 = nn.LayerNorm(token_dim)
        self.ffn = nn.Sequential(
            nn.Linear(token_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout_rate),
            nn.Linear(hidden_dim, token_dim),
            nn.Dropout(dropout_rate),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # 标准 Transformer block：self-attn -> residual+norm -> FFN -> residual+norm。
        attn_out, _ = self.self_attn(x, x, x, need_weights=False)
        x = self.norm1(x + attn_out)
        ffn_out = self.ffn(x)
        x = self.norm2(x + ffn_out)
        return x


class RankMixerBackbone(TorchModuleBase):
    def __init__(
        self,
        emb_dim: int,
        feature_preset: FeaturePreset,
        token_dim: int,
        num_layers: int = 2,
        num_heads: int = 4,
        expansion_ratio: int = 4,
        dropout_rate: float = 0.0,
        verbose: bool = False,
    ):
        super().__init__()
        assert token_dim % num_heads == 0, "token_dim % num_heads must equal 0."
        self.tokenization = FieldwiseTokenization(
            emb_dim=emb_dim,
            token_dim=token_dim,
            feature_preset=feature_preset,
            verbose=verbose,
        )
        self.layers = nn.ModuleList(
            [
                SelfAttentionMixerLayer(
                    token_dim=token_dim,
                    num_heads=num_heads,
                    expansion_ratio=expansion_ratio,
                    dropout_rate=dropout_rate,
                )
                for _ in range(num_layers)
            ]
        )
        self.output_norm = nn.LayerNorm(token_dim)

    def forward(
        self,
        batch: Dict[str, torch.Tensor],
        sparse_embeddings: Dict[str, torch.Tensor],
        hist_repr: torch.Tensor,
    ) -> torch.Tensor:
        tokens = self.tokenization(batch, sparse_embeddings, hist_repr)
        for layer in self.layers:
            tokens = layer(tokens)
        tokens = self.output_norm(tokens)
        return tokens.mean(dim=1)


In [46]:
from __future__ import annotations

# ============================================================================
# 输出 head、模型工厂与 RankMixer 参数量搜索
# ============================================================================
class JRCOutputHead(TorchModuleBase):
    def __init__(self, input_dim: int, model_config: Dict[str, Any], head_type: str):
        super().__init__()
        self.head_type = head_type
        self.logit_head, logit_hidden_dim = make_mlp(
            input_dim,
            model_config.get("logit_head_units", [64]),
            dropout_rate=model_config.get("dropout_rate", 0.0),
            output_dim=None,
        )
        out_dim = 2 if head_type == "two_logit_jrc" else 1
        self.logits_out = nn.Linear(logit_hidden_dim, out_dim)

    def forward(self, shared_repr: torch.Tensor) -> torch.Tensor:
        logits_hidden = self.logit_head(shared_repr)
        output = self.logits_out(logits_hidden)
        if self.head_type == "single_logit":
            return output.squeeze(-1)
        return output


class JRCWithMLPBackbone(TorchModuleBase):
    def __init__(
        self,
        sparse_vocab_sizes: Dict[str, int],
        dense_features: Sequence[str],
        model_config: Dict[str, Any],
        head_type: str,
    ):
        super().__init__()
        self.input_encoder = JRCBackboneInputEncoder(sparse_vocab_sizes, dense_features, model_config)
        shared_input_dim = (
            len(sparse_vocab_sizes) * model_config["emb_dim"]
            + len(dense_features)
            + model_config["emb_dim"]
        )
        self.shared_backbone, shared_output_dim = make_mlp(
            shared_input_dim,
            model_config.get("shared_dnn_units", [256, 128]),
            dropout_rate=model_config.get("dropout_rate", 0.0),
            output_dim=None,
        )
        self.output_head = JRCOutputHead(shared_output_dim, model_config, head_type=head_type)

    def forward(self, batch: Dict[str, torch.Tensor]) -> torch.Tensor:
        encoded = self.input_encoder(batch)
        shared_repr = self.shared_backbone(encoded["flat_input"])
        return self.output_head(shared_repr)


class PyTorchJRCWithRankMixerBackbone(TorchModuleBase):
    def __init__(
        self,
        sparse_vocab_sizes: Dict[str, int],
        feature_preset: FeaturePreset,
        model_config: Dict[str, Any],
        rankmixer_config: Dict[str, Any],
        head_type: str,
        verbose: bool = False,
    ):
        super().__init__()
        self.input_encoder = JRCBackboneInputEncoder(
            sparse_vocab_sizes,
            dense_features=feature_preset.model_dense_features,
            model_config=model_config,
        )
        self.shared_backbone = RankMixerBackbone(
            emb_dim=model_config["emb_dim"],
            feature_preset=feature_preset,
            token_dim=rankmixer_config["token_dim"],
            num_layers=rankmixer_config["num_layers"],
            num_heads=rankmixer_config["num_heads"],
            expansion_ratio=rankmixer_config["expansion_ratio"],
            dropout_rate=model_config.get("dropout_rate", 0.0),
            verbose=verbose,
        )
        self.output_head = JRCOutputHead(
            rankmixer_config["token_dim"],
            model_config=model_config,
            head_type=head_type,
        )

    def forward(self, batch: Dict[str, torch.Tensor]) -> torch.Tensor:
        encoded = self.input_encoder(batch)
        shared_repr = self.shared_backbone(
            batch=batch,
            sparse_embeddings=encoded["sparse_embeddings"],
            hist_repr=encoded["hist_repr"],
        )
        return self.output_head(shared_repr)


def count_parameters(model: nn.Module) -> int:
    return sum(param.numel() for param in model.parameters() if param.requires_grad)


def build_baseline_model(
    sparse_vocab_sizes: Dict[str, int],
    dense_features: Sequence[str],
    model_config: Dict[str, Any],
    head_type: str,
    device: Optional[torch.device] = None,
) -> nn.Module:
    ensure_torch_available()
    model = JRCWithMLPBackbone(
        sparse_vocab_sizes=sparse_vocab_sizes,
        dense_features=dense_features,
        model_config=model_config,
        head_type=head_type,
    )
    if device is not None:
        model = model.to(device)
    return model


def build_rankmixer_model(
    sparse_vocab_sizes: Dict[str, int],
    feature_preset: FeaturePreset,
    model_config: Dict[str, Any],
    rankmixer_config: Dict[str, Any],
    head_type: str,
    device: Optional[torch.device] = None,
    verbose: bool = False,
) -> nn.Module:
    ensure_torch_available()
    model = PyTorchJRCWithRankMixerBackbone(
        sparse_vocab_sizes=sparse_vocab_sizes,
        feature_preset=feature_preset,
        model_config=model_config,
        rankmixer_config=rankmixer_config,
        head_type=head_type,
        verbose=verbose,
    )
    if device is not None:
        model = model.to(device)
    return model


def search_rankmixer_config(
    target_params: int,
    sparse_vocab_sizes: Dict[str, int],
    feature_preset: FeaturePreset,
    model_config: Dict[str, Any],
    head_type: str,
    search_space: Dict[str, Any] = DEFAULT_RANKMIXER_SEARCH_SPACE,
) -> Dict[str, Any]:
    # 穷举搜索 token_dim / num_layers / expansion_ratio / num_heads，
    # 目标是让 RankMixer 参数量尽量贴近参考参数量。
    candidates: List[Dict[str, Any]] = []
    for token_dim in search_space["token_dim"]:
        for num_layers in search_space["num_layers"]:
            for expansion_ratio in search_space["expansion_ratio"]:
                for num_heads in search_space["num_heads"]:
                    if token_dim % num_heads != 0:
                        continue
                    candidate_config = {
                        "token_dim": token_dim,
                        "num_layers": num_layers,
                        "expansion_ratio": expansion_ratio,
                        "num_heads": num_heads,
                    }
                    model = build_rankmixer_model(
                        sparse_vocab_sizes=sparse_vocab_sizes,
                        feature_preset=feature_preset,
                        model_config=model_config,
                        rankmixer_config=candidate_config,
                        head_type=head_type,
                        device=None,
                        verbose=False,
                    )
                    params = count_parameters(model)
                    ratio = params / target_params
                    within_10pct = abs(ratio - 1.0) <= PARAM_RATIO_TOLERANCE
                    candidates.append(
                        {
                            "config": candidate_config,
                            "total_trainable_params": params,
                            "param_diff": abs(params - target_params),
                            "params_ratio_vs_reference": ratio,
                            "within_10pct": within_10pct,
                        }
                    )
                    del model

    candidates.sort(
        key=lambda item: (
            not item["within_10pct"],
            item["param_diff"],
            abs(item["params_ratio_vs_reference"] - 1.0),
            item["config"]["num_layers"],
            item["config"]["token_dim"],
            item["config"]["expansion_ratio"],
            item["config"]["num_heads"],
        )
    )
    best = candidates[0]
    within_count = sum(1 for item in candidates if item["within_10pct"])
    selection_reason = (
        "Found a RankMixer config within the +/-10% parameter budget."
        if best["within_10pct"]
        else "No candidate landed inside the +/-10% band; selected the closest parameter match available."
    )
    return {
        "config": best["config"],
        "rankmixer_params": best["total_trainable_params"],
        "params_ratio_vs_reference": best["params_ratio_vs_reference"],
        "within_10pct": best["within_10pct"],
        "selection_reason": selection_reason,
        "within_10pct_count": within_count,
        "candidates": candidates,
        "candidates_df": pd.DataFrame(
            [
                {
                    "token_dim": item["config"]["token_dim"],
                    "num_layers": item["config"]["num_layers"],
                    "expansion_ratio": item["config"]["expansion_ratio"],
                    "num_heads": item["config"]["num_heads"],
                    "total_trainable_params": item["total_trainable_params"],
                    "param_diff": item["param_diff"],
                    "params_ratio_vs_reference": item["params_ratio_vs_reference"],
                    "within_10pct": item["within_10pct"],
                }
                for item in candidates
            ]
        ),
    }


## 训练、评估与损失函数


In [47]:
from __future__ import annotations

# ============================================================================
# DataLoader：按 query_id 分组取 batch，保证 listwise loss 的组内完整性
# ============================================================================
class RankingDictDataset(TorchDatasetBase):
    def __init__(self, x_dict: Dict[str, np.ndarray], y: Optional[np.ndarray] = None):
        self.x_dict = x_dict
        self.y = y
        self.length = len(next(iter(x_dict.values())))

    def __len__(self) -> int:
        return self.length

    def __getitem__(self, idx: int):
        sample = {key: value[idx] for key, value in self.x_dict.items()}
        if self.y is None:
            return sample
        return sample, self.y[idx]


class UserGroupedBatchSampler:
    def __init__(
        self,
        query_ids: Sequence[int],
        batch_size: int = 256,
        shuffle: bool = True,
        seed: Optional[int] = None,
    ):
        self.query_ids = np.asarray(query_ids)
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.seed = seed
        self.epoch = 0
        query_to_indices: Dict[int, List[int]] = defaultdict(list)
        for idx, qid in enumerate(self.query_ids):
            query_to_indices[int(qid)].append(idx)
        self.query_blocks = list(query_to_indices.values())

    def __iter__(self):
        blocks = list(self.query_blocks)
        if self.shuffle:
            if self.seed is None:
                random.shuffle(blocks)
            else:
                rng = random.Random(self.seed + self.epoch)
                rng.shuffle(blocks)
        self.epoch += 1

        batch: List[int] = []
        for block in blocks:
            if len(block) >= self.batch_size:
                if batch:
                    yield batch
                    batch = []
                yield block
                continue
            if len(batch) + len(block) > self.batch_size:
                if batch:
                    yield batch
                batch = list(block)
            else:
                batch.extend(block)
        if batch:
            yield batch

    def __len__(self) -> int:
        count = 0
        batch: List[int] = []
        for block in self.query_blocks:
            if len(block) >= self.batch_size:
                if batch:
                    count += 1
                    batch = []
                count += 1
                continue
            if len(batch) + len(block) > self.batch_size:
                count += 1
                batch = list(block)
            else:
                batch.extend(block)
        if batch:
            count += 1
        return count


def _stack_batch_values(values: Sequence[Any]) -> np.ndarray:
    return np.stack(values)


def _torch_dtype_for_array(arr: np.ndarray):
    if np.issubdtype(arr.dtype, np.integer):
        return torch.long
    return torch.float32


def collate_with_labels(batch):
    features, labels = zip(*batch)
    collated = {}
    for key in features[0]:
        arr = _stack_batch_values([item[key] for item in features])
        collated[key] = torch.tensor(arr, dtype=_torch_dtype_for_array(arr))
    return collated, torch.tensor(np.asarray(labels), dtype=torch.float32)


def collate_features(batch):
    collated = {}
    for key in batch[0]:
        arr = _stack_batch_values([item[key] for item in batch])
        collated[key] = torch.tensor(arr, dtype=_torch_dtype_for_array(arr))
    return collated


def make_loader(
    x_dict: Dict[str, np.ndarray],
    y: Optional[np.ndarray] = None,
    batch_size: int = 256,
    shuffle: bool = False,
    group_by_query: bool = False,
    sampler_seed: Optional[int] = None,
    query_meta_key: str = "query_id",
) -> DataLoader:
    ensure_torch_available()
    dataset = RankingDictDataset(x_dict, y)
    if y is None:
        return DataLoader(dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_features)
    if group_by_query:
        batch_sampler = UserGroupedBatchSampler(
            x_dict[query_meta_key],
            batch_size=batch_size,
            shuffle=shuffle,
            seed=sampler_seed,
        )
        return DataLoader(dataset, batch_sampler=batch_sampler, collate_fn=collate_with_labels)
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, collate_fn=collate_with_labels)


def move_batch_to_device(batch: Dict[str, torch.Tensor], device: torch.device) -> Dict[str, torch.Tensor]:
    return {key: value.to(device) for key, value in batch.items()}


def debug_first_batch(train_loader) -> None:
    try:
        first_batch = next(iter(train_loader))
    except StopIteration:
        print("train_loader is empty; no batch to inspect.")
        return

    if isinstance(first_batch, tuple):
        batch_x, batch_y = first_batch
        print(f"batch_y: shape={tuple(batch_y.shape)}, dtype={batch_y.dtype}")
    else:
        batch_x = first_batch
        print("batch_y: None")

    for key, value in batch_x.items():
        if hasattr(value, "shape") and hasattr(value, "dtype"):
            print(f"{key}: shape={tuple(value.shape)}, dtype={value.dtype}")
        else:
            print(f"{key}: type={type(value).__name__}")


In [48]:
from __future__ import annotations

# ============================================================================
# 评估指标与损失函数
# ============================================================================
def safe_auc(labels: Sequence[float], preds: Sequence[float]) -> float:
    labels = np.asarray(labels)
    if np.unique(labels).size < 2:
        return float("nan")
    return float(roc_auc_score(labels, preds))


def compute_ndcg_at_k(
    labels: Sequence[float],
    preds: Sequence[float],
    query_id_list: Sequence[int],
    k: int = 5,
) -> float:
    group_score: Dict[int, List[float]] = defaultdict(list)
    group_truth: Dict[int, List[float]] = defaultdict(list)
    for idx, qid in enumerate(query_id_list):
        group_score[int(qid)].append(float(preds[idx]))
        group_truth[int(qid)].append(float(labels[idx]))
    ndcg_list: List[float] = []
    for qid in group_score:
        y_true = np.array([group_truth[qid]])
        y_score = np.array([group_score[qid]])
        if np.sum(y_true) == 0:
            ndcg_list.append(0.0)
        else:
            ndcg_list.append(float(ndcg_score(y_true, y_score, k=k)))
    return float(np.mean(ndcg_list))


def compute_mrr_hr_stats(df: pd.DataFrame, topk: int = 5, score_col: str = "pred_score") -> Dict[str, float]:
    df_sorted = df.sort_values(by=["query_id", score_col], ascending=[True, False])
    mrr_sum = 0.0
    hr_sum = 0.0
    query_count = 0
    pos_query_count = 0
    for _, group in df_sorted.groupby("query_id"):
        query_count += 1
        labels = group["label"].values
        if labels.sum() == 0:
            continue
        pos_query_count += 1
        pos_idx = np.where(labels == 1)[0]
        if len(pos_idx) > 0:
            rank = int(pos_idx[0]) + 1
            if rank <= topk:
                mrr_sum += 1.0 / rank
                hr_sum += 1.0
    return {
        "mrr5": float(mrr_sum / query_count if query_count > 0 else 0.0),
        "hr5": float(hr_sum / query_count if query_count > 0 else 0.0),
        "pos_query_count": int(pos_query_count),
        "query_count": int(query_count),
    }


def evaluate_prediction_slice(
    labels: Sequence[float],
    preds: Sequence[float],
    eval_df: pd.DataFrame,
    topk: int = 5,
) -> Dict[str, float]:
    eval_frame = eval_df[["query_id", "click_article_id", "label"]].copy()
    eval_frame["pred_score"] = preds
    rank_metrics = compute_mrr_hr_stats(eval_frame, topk=topk)
    return {
        "auc": safe_auc(labels, preds),
        "ndcg5": compute_ndcg_at_k(labels, preds, eval_df["query_id"].values, k=topk),
        "mrr5": rank_metrics["mrr5"],
        "hr5": rank_metrics["hr5"],
        "pos_query_count": rank_metrics["pos_query_count"],
        "query_count": rank_metrics["query_count"],
    }


def build_experiment_pred_score(raw_outputs: np.ndarray, head_type: str, score_mode: str = "diff") -> np.ndarray:
    if head_type == "two_logit_jrc":
        logits_2d = np.asarray(raw_outputs)
        if score_mode == "diff":
            return logits_2d[:, 1] - logits_2d[:, 0]
        if score_mode == "click":
            return logits_2d[:, 1]
        raise ValueError(f"Unknown score_mode: {score_mode}")
    if head_type == "single_logit":
        return np.asarray(raw_outputs).reshape(-1)
    raise ValueError(f"Unknown head_type: {head_type}")


def evaluate_experiment_from_raw(
    raw_outputs: np.ndarray,
    y_true: np.ndarray,
    val_df: pd.DataFrame,
    head_type: str,
    val_df_hit: Optional[pd.DataFrame] = None,
    val_hit_mask: Optional[pd.Series] = None,
    topk: int = 5,
    score_mode: str = "diff",
) -> Dict[str, Any]:
    preds = build_experiment_pred_score(raw_outputs, head_type=head_type, score_mode=score_mode)
    metrics: Dict[str, Any] = {"preds": preds, "raw_outputs": raw_outputs}

    full_metrics = evaluate_prediction_slice(y_true, preds, val_df, topk=topk)
    for key, value in full_metrics.items():
        metrics[f"full_{key}"] = value

    if val_df_hit is not None and len(val_df_hit) > 0 and val_hit_mask is not None:
        hit_mask_np = val_hit_mask.to_numpy() if hasattr(val_hit_mask, "to_numpy") else np.asarray(val_hit_mask)
        hit_preds = preds[hit_mask_np]
        hit_labels = val_df_hit["label"].values
        hit_metrics = evaluate_prediction_slice(hit_labels, hit_preds, val_df_hit, topk=topk)
        metrics["hit_preds"] = hit_preds
        for key, value in hit_metrics.items():
            metrics[f"hit_{key}"] = value
    else:
        for key in ["auc", "ndcg5", "mrr5", "hr5"]:
            metrics[f"hit_{key}"] = float("nan")
        metrics["hit_pos_query_count"] = 0
        metrics["hit_query_count"] = 0
    return metrics


def paper_context_ge_loss_old(logits_2d: torch.Tensor, labels_onehot: torch.Tensor, context_ids: torch.Tensor) -> torch.Tensor:
    batch_size = logits_2d.size(0)
    if batch_size == 0:
        return logits_2d.new_tensor(0.0)

    mask = context_ids.unsqueeze(0).eq(context_ids.unsqueeze(1))
    mask_f = mask.float()
    logits_tile = logits_2d.unsqueeze(1).expand(-1, batch_size, -1).clone()
    labels_tile = labels_onehot.unsqueeze(1).expand(-1, batch_size, -1).float()
    invalid_mask = ~mask.unsqueeze(-1)
    logits_tile = logits_tile.masked_fill(invalid_mask, -1e9)
    labels_tile = labels_tile * mask.unsqueeze(-1).float()

    l_neg = logits_tile[:, :, 0]
    l_pos = logits_tile[:, :, 1]
    y_neg = labels_tile[:, :, 0]
    y_pos = labels_tile[:, :, 1]

    log_softmax_pos = F.log_softmax(l_pos, dim=0)
    log_softmax_neg = F.log_softmax(l_neg, dim=0)

    loss_pos = -(y_pos * log_softmax_pos).sum(dim=0)
    loss_neg = -(y_neg * log_softmax_neg).sum(dim=0)

    context_sizes = mask_f.sum(dim=0).clamp_min(1.0)
    return ((loss_pos + loss_neg) / context_sizes).mean()


def paper_context_ge_loss_balanced(
    logits_2d: torch.Tensor,
    labels_onehot: torch.Tensor,
    context_ids: torch.Tensor,
    pos_weight: float = 1.0,
    neg_weight: float = 1.0,
) -> torch.Tensor:
    batch_size = logits_2d.size(0)
    if batch_size == 0:
        return logits_2d.new_tensor(0.0)

    mask = context_ids.unsqueeze(0).eq(context_ids.unsqueeze(1))
    logits_tile = logits_2d.unsqueeze(1).expand(-1, batch_size, -1).clone()
    labels_tile = labels_onehot.unsqueeze(1).expand(-1, batch_size, -1).float()
    invalid_mask = ~mask.unsqueeze(-1)
    logits_tile = logits_tile.masked_fill(invalid_mask, -1e9)
    labels_tile = labels_tile * mask.unsqueeze(-1).float()

    l_neg = logits_tile[:, :, 0]
    l_pos = logits_tile[:, :, 1]
    y_neg = labels_tile[:, :, 0]
    y_pos = labels_tile[:, :, 1]

    log_softmax_pos = F.log_softmax(l_pos, dim=0)
    log_softmax_neg = F.log_softmax(l_neg, dim=0)

    loss_pos = -(y_pos * log_softmax_pos).sum(dim=0)
    loss_neg = -(y_neg * log_softmax_neg).sum(dim=0)

    pos_cnt = y_pos.sum(dim=0).clamp_min(1.0)
    neg_cnt = y_neg.sum(dim=0).clamp_min(1.0)
    return (pos_weight * loss_pos / pos_cnt + neg_weight * loss_neg / neg_cnt).mean()


def compute_joint_loss_ge_old(
    logits_2d: torch.Tensor,
    y_true: torch.Tensor,
    context_ids: torch.Tensor,
    alpha: float = 0.5,
):
    ce_loss = F.cross_entropy(logits_2d, y_true.long())
    y_true_long = y_true.long()
    y_neg = 1 - y_true_long
    labels_onehot = torch.stack([y_neg, y_true_long], dim=1).float()
    ge_loss = paper_context_ge_loss_old(logits_2d, labels_onehot, context_ids)
    total_loss = alpha * ce_loss + (1.0 - alpha) * ge_loss
    return total_loss, ce_loss, ge_loss


def compute_joint_loss_ge_balanced(
    logits_2d: torch.Tensor,
    y_true: torch.Tensor,
    context_ids: torch.Tensor,
    alpha: float = 0.5,
    pos_weight: float = 1.0,
    neg_weight: float = 1.0,
):
    ce_loss = F.cross_entropy(logits_2d, y_true.long())
    y_true_long = y_true.long()
    y_neg = 1 - y_true_long
    labels_onehot = torch.stack([y_neg, y_true_long], dim=1).float()
    ge_loss = paper_context_ge_loss_balanced(
        logits_2d,
        labels_onehot,
        context_ids,
        pos_weight=pos_weight,
        neg_weight=neg_weight,
    )
    total_loss = alpha * ce_loss + (1.0 - alpha) * ge_loss
    return total_loss, ce_loss, ge_loss


def compute_query_softmax_ce(
    scores: torch.Tensor,
    y_true: torch.Tensor,
    query_ids: torch.Tensor,
) -> torch.Tensor:
    # 每个 query 只有一个正样本时，这里就是题目要求的 query-wise softmax CE。
    unique_query_ids = torch.unique(query_ids)
    group_losses: List[torch.Tensor] = []
    for query_id in unique_query_ids:
        mask = query_ids.eq(query_id)
        group_scores = scores[mask]
        group_labels = y_true[mask]
        pos_idx = torch.nonzero(group_labels > 0.5, as_tuple=False).flatten()
        if pos_idx.numel() == 0:
            continue
        group_log_probs = F.log_softmax(group_scores, dim=0)
        group_losses.append(-group_log_probs[pos_idx].mean())
    if not group_losses:
        return scores.new_tensor(0.0)
    return torch.stack(group_losses).mean()


In [49]:
from __future__ import annotations

# ============================================================================
# 训练循环：2D-logit JRC 主线 + single-logit listwise 消融
# ============================================================================
def raw_predict_outputs(
    model: nn.Module,
    x_data: Dict[str, np.ndarray],
    batch_size: int = 256,
    device: Optional[torch.device] = None,
) -> np.ndarray:
    ensure_torch_available()
    if device is None:
        device = get_default_device()
    loader = make_loader(x_data, y=None, batch_size=batch_size, shuffle=False)
    model.eval()
    outputs: List[np.ndarray] = []
    with torch.no_grad():
        for batch in loader:
            batch = move_batch_to_device(batch, device)
            raw_output = model(batch)
            outputs.append(raw_output.detach().cpu().numpy())
    return np.concatenate(outputs, axis=0)


def train_ranking_model(
    model: nn.Module,
    x_train: Dict[str, np.ndarray],
    y_train: np.ndarray,
    head_type: str,
    loss_type: str,
    score_mode: str,
    x_val: Optional[Dict[str, np.ndarray]] = None,
    y_val: Optional[np.ndarray] = None,
    val_df: Optional[pd.DataFrame] = None,
    val_df_hit: Optional[pd.DataFrame] = None,
    val_hit_mask: Optional[pd.Series] = None,
    epochs: int = 5,
    batch_size: int = 256,
    lr: float = 1e-3,
    select_topk: int = 5,
    alpha: float = 0.5,
    pos_weight: float = 1.0,
    neg_weight: float = 1.0,
    seed: Optional[int] = None,
    device: Optional[torch.device] = None,
) -> Dict[str, Any]:
    ensure_torch_available()
    if device is None:
        device = get_default_device()
    if seed is not None:
        set_global_seed(seed)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    train_loader = make_loader(
        x_train,
        y_train,
        batch_size=batch_size,
        shuffle=True,
        group_by_query=True,
        sampler_seed=seed,
        query_meta_key="query_id",
    )

    history: Dict[str, List[float]] = {
        "total_loss": [],
        "ce_loss": [],
        "ge_loss": [],
        "query_softmax_ce": [],
        "full_auc": [],
        "full_ndcg5": [],
        "full_mrr5": [],
        "full_hr5": [],
        "hit_auc": [],
        "hit_ndcg5": [],
        "hit_mrr5": [],
        "hit_hr5": [],
    }
    best_record = {
        "metric": float("-inf"),
        "epoch": None,
        "state_dict": None,
        "eval_metrics": None,
    }

    for epoch in range(epochs):
        epoch_losses = {key: [] for key in ["total_loss", "ce_loss", "ge_loss", "query_softmax_ce"]}
        t0 = time.time()
        model.train()

        for x_batch, y_batch in train_loader:
            x_batch = move_batch_to_device(x_batch, device)
            y_batch = y_batch.to(device)
            query_ids = x_batch["query_id"]

            optimizer.zero_grad()
            raw_output = model(x_batch)

            # 两种 head 共用同一套输入与 backbone，只在 loss / 输出解释上分流。
            if head_type == "two_logit_jrc":
                logits_2d = raw_output
                if loss_type == "old_ge":
                    total_loss, ce_loss, ge_loss = compute_joint_loss_ge_old(
                        logits_2d,
                        y_batch,
                        query_ids,
                        alpha=alpha,
                    )
                elif loss_type == "balanced_ge":
                    total_loss, ce_loss, ge_loss = compute_joint_loss_ge_balanced(
                        logits_2d,
                        y_batch,
                        query_ids,
                        alpha=alpha,
                        pos_weight=pos_weight,
                        neg_weight=neg_weight,
                    )
                else:
                    raise ValueError(f"Unknown two_logit_jrc loss_type: {loss_type}")
                query_softmax_ce = logits_2d.new_tensor(0.0)
            elif head_type == "single_logit":
                if loss_type not in {"query_softmax_ce", "single_logit_listwise"}:
                    raise ValueError(f"Unknown single_logit loss_type: {loss_type}")
                scores = raw_output.reshape(-1)
                query_softmax_ce = compute_query_softmax_ce(scores, y_batch, query_ids)
                total_loss = query_softmax_ce
                ce_loss = query_softmax_ce
                ge_loss = scores.new_tensor(0.0)
            else:
                raise ValueError(f"Unknown head_type: {head_type}")

            total_loss.backward()
            optimizer.step()

            epoch_losses["total_loss"].append(float(total_loss.item()))
            epoch_losses["ce_loss"].append(float(ce_loss.item()))
            epoch_losses["ge_loss"].append(float(ge_loss.item()))
            epoch_losses["query_softmax_ce"].append(float(query_softmax_ce.item()))

        elapsed = time.time() - t0
        for loss_name in epoch_losses:
            history[loss_name].append(float(np.mean(epoch_losses[loss_name])))

        log_str = (
            f"Epoch {epoch + 1}/{epochs} ({elapsed:.1f}s)  "
            f"Total: {history['total_loss'][-1]:.4f}  "
            f"CE: {history['ce_loss'][-1]:.4f}  "
            f"GE: {history['ge_loss'][-1]:.4f}"
        )
        if head_type == "single_logit":
            log_str += f"  | query_softmax_ce: {history['query_softmax_ce'][-1]:.4f}"

        if x_val is not None and y_val is not None and val_df is not None:
            val_raw_outputs = raw_predict_outputs(model, x_val, batch_size=batch_size, device=device)
            eval_metrics = evaluate_experiment_from_raw(
                val_raw_outputs,
                y_val,
                val_df,
                head_type=head_type,
                val_df_hit=val_df_hit,
                val_hit_mask=val_hit_mask,
                topk=select_topk,
                score_mode=score_mode,
            )
            for metric_name in [
                "full_auc",
                "full_ndcg5",
                "full_mrr5",
                "full_hr5",
                "hit_auc",
                "hit_ndcg5",
                "hit_mrr5",
                "hit_hr5",
            ]:
                history[metric_name].append(float(eval_metrics.get(metric_name, float("nan"))))

            current_metric = eval_metrics["hit_mrr5"]
            if np.isnan(current_metric):
                current_metric = eval_metrics["full_mrr5"]
            if current_metric > best_record["metric"]:
                best_record["metric"] = float(current_metric)
                best_record["epoch"] = epoch + 1
                best_record["state_dict"] = copy.deepcopy(model.state_dict())
                best_record["eval_metrics"] = {
                    key: value
                    for key, value in eval_metrics.items()
                    if key not in {"preds", "hit_preds", "raw_outputs"}
                }
                log_str += "  | best*"

            log_str += (
                f"  | Full MRR@{select_topk}: {eval_metrics['full_mrr5']:.5f}"
                f"  | Hit MRR@{select_topk}: {eval_metrics['hit_mrr5']:.5f}"
                f"  | Full NDCG@{select_topk}: {eval_metrics['full_ndcg5']:.4f}"
                f"  | Hit NDCG@{select_topk}: {eval_metrics['hit_ndcg5']:.4f}"
            )

        print(log_str)

    if best_record["state_dict"] is not None:
        model.load_state_dict(best_record["state_dict"])

    return {
        "history": history,
        "best_record": {
            "metric": None if best_record["metric"] == float("-inf") else float(best_record["metric"]),
            "epoch": best_record["epoch"],
            "state_dict": best_record["state_dict"],
            "eval_metrics": best_record["eval_metrics"],
        },
    }


## 实验编排


In [50]:
from __future__ import annotations

# ============================================================================
# 实验编排与结果汇总
# ============================================================================
def make_run_record(
    scenario: PreparedScenario,
    model_name: str,
    backbone_type: str,
    loss_type: str,
    score_mode: str,
    total_trainable_params: int,
    train_result: Dict[str, Any],
) -> Dict[str, Any]:
    best_record = train_result["best_record"]
    eval_metrics = best_record.get("eval_metrics", {}) or {}
    return {
        "scenario_name": scenario.spec.name,
        "feature_preset": scenario.spec.feature_preset.name,
        "feature_description": scenario.spec.feature_preset.description,
        "head_type": scenario.spec.head_type,
        "is_mainline": scenario.spec.is_mainline,
        "scenario_notes": scenario.spec.notes,
        "model_name": model_name,
        "backbone_type": backbone_type,
        "loss_type": loss_type,
        "score_mode": score_mode,
        "total_trainable_params": int(total_trainable_params),
        "reference_params": int(scenario.reference_params),
        "params_ratio_vs_reference": float(total_trainable_params / scenario.reference_params),
        "rankmixer_within_10pct": bool(scenario.rankmixer_search_result["within_10pct"]),
        "selected_rankmixer_reason": scenario.rankmixer_search_result["selection_reason"],
        "best_epoch": best_record.get("epoch"),
        "best_hit_mrr5": eval_metrics.get("hit_mrr5", float("nan")),
        "best_full_mrr5": eval_metrics.get("full_mrr5", float("nan")),
        "best_hit_ndcg5": eval_metrics.get("hit_ndcg5", float("nan")),
        "best_full_ndcg5": eval_metrics.get("full_ndcg5", float("nan")),
        "best_hit_hr5": eval_metrics.get("hit_hr5", float("nan")),
        "best_full_hr5": eval_metrics.get("full_hr5", float("nan")),
        "full_auc_at_best": eval_metrics.get("full_auc", float("nan")),
        "hit_auc_at_best": eval_metrics.get("hit_auc", float("nan")),
        "history": train_result["history"],
        "best_record": best_record,
        "model_config": {
            "common": copy.deepcopy(scenario.model_config),
            "rankmixer": copy.deepcopy(scenario.selected_rankmixer_config),
        },
    }


def _selection_metric(record: Dict[str, Any]) -> float:
    hit_mrr5 = record.get("best_hit_mrr5", float("nan"))
    full_mrr5 = record.get("best_full_mrr5", float("nan"))
    if hit_mrr5 is not None and not np.isnan(hit_mrr5):
        return float(hit_mrr5)
    if full_mrr5 is not None and not np.isnan(full_mrr5):
        return float(full_mrr5)
    return float("-inf")


def flatten_run_record(record: Dict[str, Any]) -> Dict[str, Any]:
    return {
        "scenario_name": record["scenario_name"],
        "feature_preset": record["feature_preset"],
        "head_type": record["head_type"],
        "is_mainline": record["is_mainline"],
        "model_name": record["model_name"],
        "backbone_type": record["backbone_type"],
        "loss_type": record["loss_type"],
        "score_mode": record["score_mode"],
        "total_trainable_params": record["total_trainable_params"],
        "reference_params": record["reference_params"],
        "params_ratio_vs_reference": record["params_ratio_vs_reference"],
        "best_epoch": record["best_epoch"],
        "best_hit_mrr5": record["best_hit_mrr5"],
        "best_full_mrr5": record["best_full_mrr5"],
        "best_hit_ndcg5": record["best_hit_ndcg5"],
        "best_full_ndcg5": record["best_full_ndcg5"],
        "best_hit_hr5": record["best_hit_hr5"],
        "best_full_hr5": record["best_full_hr5"],
        "full_auc_at_best": record["full_auc_at_best"],
        "hit_auc_at_best": record["hit_auc_at_best"],
        "rankmixer_within_10pct": record["rankmixer_within_10pct"],
        "selected_rankmixer_reason": record["selected_rankmixer_reason"],
    }


def make_comparison_dataframe(records: Sequence[Dict[str, Any]]) -> pd.DataFrame:
    rows = [flatten_run_record(record) for record in records]
    compare_df = pd.DataFrame(rows)
    if len(compare_df) == 0:
        return compare_df
    compare_df["selection_metric"] = [_selection_metric(record) for record in records]
    compare_df = compare_df.sort_values(
        ["selection_metric", "best_full_mrr5", "scenario_name"],
        ascending=[False, False, True],
    ).reset_index(drop=True)
    return compare_df.drop(columns=["selection_metric"])


def select_best_record(
    records: Sequence[Dict[str, Any]],
    backbone_type: Optional[str] = None,
    scenario_name: Optional[str] = None,
) -> Optional[Dict[str, Any]]:
    filtered = [
        record
        for record in records
        if (backbone_type is None or record["backbone_type"] == backbone_type)
        and (scenario_name is None or record["scenario_name"] == scenario_name)
    ]
    if not filtered:
        return None
    return max(filtered, key=_selection_metric)


def make_scenario_summary_dataframe(scenarios: Sequence[PreparedScenario]) -> pd.DataFrame:
    rows: List[Dict[str, Any]] = []
    for scenario in scenarios:
        search_result = scenario.rankmixer_search_result
        rows.append(
            {
                "scenario_name": scenario.spec.name,
                "feature_preset": scenario.spec.feature_preset.name,
                "head_type": scenario.spec.head_type,
                "protocol_count": len(scenario.spec.protocols),
                "reference_params": scenario.reference_params,
                "selected_rankmixer_params": scenario.selected_rankmixer_params,
                "params_ratio_vs_reference": search_result["params_ratio_vs_reference"],
                "rankmixer_within_10pct": search_result["within_10pct"],
                "token_count": len(scenario.token_schema_df),
                "selected_rankmixer_config": copy.deepcopy(scenario.selected_rankmixer_config),
                "selection_reason": search_result["selection_reason"],
                "notes": scenario.spec.notes,
            }
        )
    return pd.DataFrame(rows)


def prepare_default_suite(
    merged_frames: MergedFeatureFrames,
    emb_dim: int = DEFAULT_EMB_DIM,
    max_len: int = DEFAULT_MAX_HIST_LEN,
    model_config: Optional[Dict[str, Any]] = None,
    search_space: Dict[str, Any] = DEFAULT_RANKMIXER_SEARCH_SPACE,
) -> PreparedAblationSuite:
    ensure_torch_available()
    model_config = copy.deepcopy(model_config or DEFAULT_MODEL_CONFIG)
    model_config["emb_dim"] = emb_dim
    model_config["max_len"] = max_len

    sparse_feature_diagnostics_df = compute_sparse_feature_overlap_diagnostics(
        merged_frames.trn_df,
        merged_frames.val_df,
    )
    scenarios: List[PreparedScenario] = []

    # 先为每个 scenario 准备输入，再为对应的 RankMixer 搜索最接近 baseline 的参数量配置。
    for scenario_spec in build_default_scenarios():
        prepared_data = prepare_model_inputs(
            merged_frames,
            feature_preset=scenario_spec.feature_preset,
            emb_dim=emb_dim,
            max_len=max_len,
        )
        baseline_model = build_baseline_model(
            sparse_vocab_sizes=prepared_data.sparse_vocab_sizes,
            dense_features=prepared_data.feature_preset.model_dense_features,
            model_config=model_config,
            head_type=scenario_spec.head_type,
            device=None,
        )
        reference_params = count_parameters(baseline_model)
        del baseline_model
        clear_torch_cache()

        search_result = search_rankmixer_config(
            target_params=reference_params,
            sparse_vocab_sizes=prepared_data.sparse_vocab_sizes,
            feature_preset=prepared_data.feature_preset,
            model_config=model_config,
            head_type=scenario_spec.head_type,
            search_space=search_space,
        )
        selected_rankmixer_config = search_result["config"]
        preview_model = build_rankmixer_model(
            sparse_vocab_sizes=prepared_data.sparse_vocab_sizes,
            feature_preset=prepared_data.feature_preset,
            model_config=model_config,
            rankmixer_config=selected_rankmixer_config,
            head_type=scenario_spec.head_type,
            device=None,
            verbose=False,
        )
        selected_rankmixer_params = count_parameters(preview_model)
        del preview_model
        clear_torch_cache()

        scenarios.append(
            PreparedScenario(
                spec=scenario_spec,
                prepared_data=prepared_data,
                model_config=copy.deepcopy(model_config),
                reference_params=reference_params,
                selected_rankmixer_config=selected_rankmixer_config,
                selected_rankmixer_params=selected_rankmixer_params,
                rankmixer_search_result=search_result,
                token_schema_df=make_token_schema_dataframe(prepared_data.feature_preset),
            )
        )

    return PreparedAblationSuite(
        merged_frames=merged_frames,
        sparse_feature_diagnostics_df=sparse_feature_diagnostics_df,
        scenarios=scenarios,
    )


def run_prepared_suite(
    suite: PreparedAblationSuite,
    epochs: int,
    batch_size: int,
    lr: float,
    select_topk: int,
    alpha: float,
    seed: int,
    device: Optional[torch.device] = None,
) -> List[Dict[str, Any]]:
    ensure_torch_available()
    if device is None:
        device = get_default_device()
    all_run_records: List[Dict[str, Any]] = []

    for scenario in suite.scenarios:
        prepared_data = scenario.prepared_data
        print("\n" + "#" * 80)
        print(f"Scenario: {scenario.spec.name}")
        print(f"Feature preset: {prepared_data.feature_preset.name}")
        print(f"Head type: {scenario.spec.head_type}")
        print(scenario.spec.notes)
        print(
            f"Reference params: {scenario.reference_params:,} | "
            f"Selected RankMixer params: {scenario.selected_rankmixer_params:,} | "
            f"Ratio vs reference: {scenario.selected_rankmixer_params / scenario.reference_params:.4f}"
        )
        print(f"RankMixer config: {scenario.selected_rankmixer_config}")
        print(f"Param matching note: {scenario.rankmixer_search_result['selection_reason']}")

        for protocol in scenario.spec.protocols:
            loss_type = protocol["loss_type"]
            score_mode = protocol["score_mode"]
            print("\n" + "=" * 70)
            print(f"Protocol: {loss_type} + {score_mode}")
            print("=" * 70)

            set_global_seed(seed)
            rankmixer_model = build_rankmixer_model(
                sparse_vocab_sizes=prepared_data.sparse_vocab_sizes,
                feature_preset=prepared_data.feature_preset,
                model_config=scenario.model_config,
                rankmixer_config=scenario.selected_rankmixer_config,
                head_type=scenario.spec.head_type,
                device=device,
                verbose=False,
            )
            rankmixer_params = count_parameters(rankmixer_model)
            print(
                f"RankMixer trainable params: {rankmixer_params:,} | "
                f"ratio vs reference params: {rankmixer_params / scenario.reference_params:.4f}"
            )
            rankmixer_result = train_ranking_model(
                rankmixer_model,
                prepared_data.x_trn,
                prepared_data.y_trn,
                head_type=scenario.spec.head_type,
                loss_type=loss_type,
                score_mode=score_mode,
                x_val=prepared_data.x_val,
                y_val=prepared_data.y_val,
                val_df=prepared_data.val_df,
                val_df_hit=prepared_data.val_df_hit,
                val_hit_mask=prepared_data.val_hit_mask,
                epochs=epochs,
                batch_size=batch_size,
                lr=lr,
                select_topk=select_topk,
                alpha=alpha,
                seed=seed,
                device=device,
            )
            all_run_records.append(
                make_run_record(
                    scenario=scenario,
                    model_name=f"{scenario.spec.name}__rankmixer__{loss_type}__{score_mode}",
                    backbone_type="rankmixer",
                    loss_type=loss_type,
                    score_mode=score_mode,
                    total_trainable_params=rankmixer_params,
                    train_result=rankmixer_result,
                )
            )
            del rankmixer_model
            clear_torch_cache()

    return all_run_records


def summarize_best_rankmixer_records(records: Sequence[Dict[str, Any]]) -> pd.DataFrame:
    rows: List[Dict[str, Any]] = []
    scenario_names = sorted({record["scenario_name"] for record in records})
    for scenario_name in scenario_names:
        best_record = select_best_record(records, backbone_type="rankmixer", scenario_name=scenario_name)
        if best_record is None:
            continue
        rows.append(flatten_run_record(best_record))
    return pd.DataFrame(rows)


## 构建实验套件


In [51]:
from __future__ import annotations

common_model_config = dict(DEFAULT_MODEL_CONFIG)
common_model_config["emb_dim"] = DEFAULT_EMB_DIM
common_model_config["max_len"] = DEFAULT_MAX_HIST_LEN

ensure_torch_available()
suite = prepare_default_suite(
    merged_frames,
    emb_dim=DEFAULT_EMB_DIM,
    max_len=DEFAULT_MAX_HIST_LEN,
    model_config=common_model_config,
)
scenario_summary_df = make_scenario_summary_dataframe(suite.scenarios)
display(scenario_summary_df)


,scenario_name,feature_preset,head_type,protocol_count,reference_params,selected_rankmixer_params,params_ratio_vs_reference,rankmixer_within_10pct,token_count,selected_rankmixer_config,selection_reason,notes
0,fair_main_two_logit_jrc,fair_main,two_logit_jrc,2,254307,254243,0.999748,True,12,"{'token_dim': 48, 'num_layers': 3, 'expansion_...",Found a RankMixer config within the +/-10% par...,Primary 2D-logit JRC line with balanced_ge + c...
1,user_id_control_two_logit_jrc,user_id_control,two_logit_jrc,2,1454435,1453715,0.999505,True,13,"{'token_dim': 80, 'num_layers': 1, 'expansion_...",Found a RankMixer config within the +/-10% par...,Control branch that restores user_id as a lear...
2,fair_main_single_logit,fair_main,single_logit,1,254242,254178,0.999748,True,12,"{'token_dim': 48, 'num_layers': 3, 'expansion_...",Found a RankMixer config within the +/-10% par...,Single-logit listwise softmax ablation. This i...


In [52]:
from __future__ import annotations

for scenario in suite.scenarios:
    print()
    print("=" * 80)
    print(f"Scenario: {scenario.spec.name}")
    print(f"Feature preset: {scenario.spec.feature_preset.name}")
    print(f"Head type: {scenario.spec.head_type}")
    print(f"Notes: {scenario.spec.notes}")
    print(f"Selected RankMixer config: {scenario.selected_rankmixer_config}")
    print(
        f"Selected RankMixer params: {scenario.selected_rankmixer_params:,} "
        f"(ratio vs reference: {scenario.selected_rankmixer_params / scenario.reference_params:.4f})"
    )
    print(f"Param matching note: {scenario.rankmixer_search_result['selection_reason']}")
    display(scenario.token_schema_df)
    display(scenario.rankmixer_search_result["candidates_df"].head(10))



Scenario: fair_main_two_logit_jrc
Feature preset: fair_main
Head type: two_logit_jrc
Notes: Primary 2D-logit JRC line with balanced_ge + click and old_ge + diff.
Selected RankMixer config: {'token_dim': 48, 'num_layers': 3, 'expansion_ratio': 2, 'num_heads': 2}
Selected RankMixer params: 254,243 (ratio vs reference: 0.9997)
Param matching note: Found a RankMixer config within the +/-10% parameter budget.


,token_index,token_name,kind,features
0,1,click_article_id,sparse,click_article_id
1,2,category_id,sparse,category_id
2,3,click_environment,sparse,click_environment
3,4,click_deviceGroup,sparse,click_deviceGroup
4,5,click_os,sparse,click_os
5,6,click_country,sparse,click_country
6,7,click_region,sparse,click_region
7,8,click_referrer_type,sparse,click_referrer_type
8,9,recall_similarity_dense,dense,"sim0, time_diff0, word_diff0, sim_max, sim_min..."
9,10,ranking_meta_dense,dense,"score, rank, click_size, time_diff_mean, activ..."


,token_dim,num_layers,expansion_ratio,num_heads,total_trainable_params,param_diff,params_ratio_vs_reference,within_10pct
0,48,3,2,2,254243,64,0.999748,True
1,48,3,2,4,254243,64,0.999748,True
2,48,3,2,6,254243,64,0.999748,True
3,48,3,2,8,254243,64,0.999748,True
4,72,1,3,2,254171,136,0.999465,True
5,72,1,3,4,254171,136,0.999465,True
6,72,1,3,6,254171,136,0.999465,True
7,72,1,3,8,254171,136,0.999465,True
8,48,2,4,2,253907,400,0.998427,True
9,48,2,4,4,253907,400,0.998427,True



Scenario: user_id_control_two_logit_jrc
Feature preset: user_id_control
Head type: two_logit_jrc
Notes: Control branch that restores user_id as a learned sparse feature under the same RankMixer-only protocols.
Selected RankMixer config: {'token_dim': 80, 'num_layers': 1, 'expansion_ratio': 2, 'num_heads': 2}
Selected RankMixer params: 1,453,715 (ratio vs reference: 0.9995)
Param matching note: Found a RankMixer config within the +/-10% parameter budget.


,token_index,token_name,kind,features
0,1,user_id_sparse_control,sparse,user_id
1,2,click_article_id,sparse,click_article_id
2,3,category_id,sparse,category_id
3,4,click_environment,sparse,click_environment
4,5,click_deviceGroup,sparse,click_deviceGroup
5,6,click_os,sparse,click_os
6,7,click_country,sparse,click_country
7,8,click_region,sparse,click_region
8,9,click_referrer_type,sparse,click_referrer_type
9,10,recall_similarity_dense,dense,"sim0, time_diff0, word_diff0, sim_max, sim_min..."


,token_dim,num_layers,expansion_ratio,num_heads,total_trainable_params,param_diff,params_ratio_vs_reference,within_10pct
0,80,1,2,2,1453715,720,0.999505,True
1,80,1,2,4,1453715,720,0.999505,True
2,80,1,2,8,1453715,720,0.999505,True
3,40,3,4,2,1453595,840,0.999422,True
4,40,3,4,4,1453595,840,0.999422,True
5,40,3,4,8,1453595,840,0.999422,True
6,72,1,3,2,1452899,1536,0.998944,True
7,72,1,3,4,1452899,1536,0.998944,True
8,72,1,3,6,1452899,1536,0.998944,True
9,72,1,3,8,1452899,1536,0.998944,True



Scenario: fair_main_single_logit
Feature preset: fair_main
Head type: single_logit
Notes: Single-logit listwise softmax ablation. This is an ablation on top of the 2D mainline, not a replacement.
Selected RankMixer config: {'token_dim': 48, 'num_layers': 3, 'expansion_ratio': 2, 'num_heads': 2}
Selected RankMixer params: 254,178 (ratio vs reference: 0.9997)
Param matching note: Found a RankMixer config within the +/-10% parameter budget.


,token_index,token_name,kind,features
0,1,click_article_id,sparse,click_article_id
1,2,category_id,sparse,category_id
2,3,click_environment,sparse,click_environment
3,4,click_deviceGroup,sparse,click_deviceGroup
4,5,click_os,sparse,click_os
5,6,click_country,sparse,click_country
6,7,click_region,sparse,click_region
7,8,click_referrer_type,sparse,click_referrer_type
8,9,recall_similarity_dense,dense,"sim0, time_diff0, word_diff0, sim_max, sim_min..."
9,10,ranking_meta_dense,dense,"score, rank, click_size, time_diff_mean, activ..."


,token_dim,num_layers,expansion_ratio,num_heads,total_trainable_params,param_diff,params_ratio_vs_reference,within_10pct
0,48,3,2,2,254178,64,0.999748,True
1,48,3,2,4,254178,64,0.999748,True
2,48,3,2,6,254178,64,0.999748,True
3,48,3,2,8,254178,64,0.999748,True
4,72,1,3,2,254106,136,0.999465,True
5,72,1,3,4,254106,136,0.999465,True
6,72,1,3,6,254106,136,0.999465,True
7,72,1,3,8,254106,136,0.999465,True
8,48,2,4,2,253842,400,0.998427,True
9,48,2,4,4,253842,400,0.998427,True


## 运行实验


In [53]:
from __future__ import annotations

all_run_records = run_prepared_suite(
    suite,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    lr=LR,
    select_topk=TOPK,
    alpha=ALPHA,
    seed=SEED,
    device=DEVICE,
)

print(f"Completed RankMixer runs: {len(all_run_records)}")



################################################################################
Scenario: fair_main_two_logit_jrc
Feature preset: fair_main
Head type: two_logit_jrc
Primary 2D-logit JRC line with balanced_ge + click and old_ge + diff.
Reference params: 254,307 | Selected RankMixer params: 254,243 | Ratio vs reference: 0.9997
RankMixer config: {'token_dim': 48, 'num_layers': 3, 'expansion_ratio': 2, 'num_heads': 2}
Param matching note: Found a RankMixer config within the +/-10% parameter budget.

Protocol: balanced_ge + click
RankMixer trainable params: 254,243 | ratio vs reference params: 0.9997
Epoch 1/5 (160.5s)  Total: 2.2321  CE: 0.2073  GE: 4.2570  | best*  | Full MRR@5: 0.17730  | Hit MRR@5: 0.25901  | Full NDCG@5: 0.2116  | Hit NDCG@5: 0.3091
Epoch 2/5 (150.1s)  Total: 2.1178  CE: 0.1905  GE: 4.0452  | best*  | Full MRR@5: 0.20234  | Hit MRR@5: 0.29558  | Full NDCG@5: 0.2386  | Hit NDCG@5: 0.3485
Epoch 3/5 (143.8s)  Total: 2.0686  CE: 0.1832  GE: 3.9539  | best*  | Full MRR@5:

## 结果汇总


In [54]:
from __future__ import annotations

rankmixer_compare_df = make_comparison_dataframe(all_run_records)
two_logit_compare_df = rankmixer_compare_df[
    rankmixer_compare_df["head_type"] == "two_logit_jrc"
].reset_index(drop=True)
single_logit_compare_df = rankmixer_compare_df[
    rankmixer_compare_df["head_type"] == "single_logit"
].reset_index(drop=True)

display(two_logit_compare_df)
display(single_logit_compare_df)
print("This notebook now compares RankMixer branches only.")
print("single-logit is an ablation on top of the 2D-logit JRC mainline, not a replacement.")


,scenario_name,feature_preset,head_type,is_mainline,model_name,backbone_type,loss_type,score_mode,total_trainable_params,reference_params,...,best_hit_mrr5,best_full_mrr5,best_hit_ndcg5,best_full_ndcg5,best_hit_hr5,best_full_hr5,full_auc_at_best,hit_auc_at_best,rankmixer_within_10pct,selected_rankmixer_reason
0,fair_main_two_logit_jrc,fair_main,two_logit_jrc,True,fair_main_two_logit_jrc__rankmixer__balanced_g...,rankmixer,balanced_ge,click,254243,254307,...,0.303027,0.207437,0.356080,0.243755,0.517165,0.354025,0.950388,0.951075,True,Found a RankMixer config within the +/-10% par...
1,fair_main_two_logit_jrc,fair_main,two_logit_jrc,True,fair_main_two_logit_jrc__rankmixer__old_ge__diff,rankmixer,old_ge,diff,254243,254307,...,0.299601,0.205092,0.350960,0.240250,0.506939,0.347025,0.945781,0.946313,True,Found a RankMixer config within the +/-10% par...
2,user_id_control_two_logit_jrc,user_id_control,two_logit_jrc,False,user_id_control_two_logit_jrc__rankmixer__bala...,rankmixer,balanced_ge,click,1453715,1454435,...,0.296132,0.202717,0.347732,0.238040,0.504346,0.345250,0.936909,0.936719,True,Found a RankMixer config within the +/-10% par...
3,user_id_control_two_logit_jrc,user_id_control,two_logit_jrc,False,user_id_control_two_logit_jrc__rankmixer__old_...,rankmixer,old_ge,diff,1453715,1454435,...,0.295009,0.201948,0.347357,0.237783,0.506501,0.346725,0.946880,0.948287,True,Found a RankMixer config within the +/-10% par...


,scenario_name,feature_preset,head_type,is_mainline,model_name,backbone_type,loss_type,score_mode,total_trainable_params,reference_params,...,best_hit_mrr5,best_full_mrr5,best_hit_ndcg5,best_full_ndcg5,best_hit_hr5,best_full_hr5,full_auc_at_best,hit_auc_at_best,rankmixer_within_10pct,selected_rankmixer_reason
0,fair_main_single_logit,fair_main,single_logit,False,fair_main_single_logit__rankmixer__query_softm...,rankmixer,query_softmax_ce,scalar,254178,254242,...,0.302185,0.206861,0.355586,0.243416,0.517603,0.354325,0.937629,0.93709,True,Found a RankMixer config within the +/-10% par...


This notebook now compares RankMixer branches only.
single-logit is an ablation on top of the 2D-logit JRC mainline, not a replacement.


In [55]:
from __future__ import annotations

best_rankmixer_df = summarize_best_rankmixer_records(all_run_records)
display(best_rankmixer_df)

for scenario in suite.scenarios:
    best_record = select_best_record(
        all_run_records,
        scenario_name=scenario.spec.name,
    )
    if best_record is None:
        continue
    print()
    print("-" * 70)
    print(f"Best RankMixer scenario: {scenario.spec.name}")
    print(
        f"Protocol: {best_record['loss_type']} + {best_record['score_mode']} | "
        f"ratio vs reference: {best_record['params_ratio_vs_reference']:.4f}"
    )
    print(f"RankMixer config: {best_record['model_config']['rankmixer']}")
    print(f"Selection note: {best_record['selected_rankmixer_reason']}")


,scenario_name,feature_preset,head_type,is_mainline,model_name,backbone_type,loss_type,score_mode,total_trainable_params,reference_params,...,best_hit_mrr5,best_full_mrr5,best_hit_ndcg5,best_full_ndcg5,best_hit_hr5,best_full_hr5,full_auc_at_best,hit_auc_at_best,rankmixer_within_10pct,selected_rankmixer_reason
0,fair_main_single_logit,fair_main,single_logit,False,fair_main_single_logit__rankmixer__query_softm...,rankmixer,query_softmax_ce,scalar,254178,254242,...,0.302185,0.206861,0.355586,0.243416,0.517603,0.354325,0.937629,0.937090,True,Found a RankMixer config within the +/-10% par...
1,fair_main_two_logit_jrc,fair_main,two_logit_jrc,True,fair_main_two_logit_jrc__rankmixer__balanced_g...,rankmixer,balanced_ge,click,254243,254307,...,0.303027,0.207437,0.356080,0.243755,0.517165,0.354025,0.950388,0.951075,True,Found a RankMixer config within the +/-10% par...
2,user_id_control_two_logit_jrc,user_id_control,two_logit_jrc,False,user_id_control_two_logit_jrc__rankmixer__bala...,rankmixer,balanced_ge,click,1453715,1454435,...,0.296132,0.202717,0.347732,0.238040,0.504346,0.345250,0.936909,0.936719,True,Found a RankMixer config within the +/-10% par...



----------------------------------------------------------------------
Best RankMixer scenario: fair_main_two_logit_jrc
Protocol: balanced_ge + click | ratio vs reference: 0.9997
RankMixer config: {'token_dim': 48, 'num_layers': 3, 'expansion_ratio': 2, 'num_heads': 2}
Selection note: Found a RankMixer config within the +/-10% parameter budget.

----------------------------------------------------------------------
Best RankMixer scenario: user_id_control_two_logit_jrc
Protocol: balanced_ge + click | ratio vs reference: 0.9995
RankMixer config: {'token_dim': 80, 'num_layers': 1, 'expansion_ratio': 2, 'num_heads': 2}
Selection note: Found a RankMixer config within the +/-10% parameter budget.

----------------------------------------------------------------------
Best RankMixer scenario: fair_main_single_logit
Protocol: query_softmax_ce + scalar | ratio vs reference: 0.9997
RankMixer config: {'token_dim': 48, 'num_layers': 3, 'expansion_ratio': 2, 'num_heads': 2}
Selection note: Found 